# Evaluation on Vezilka LLM using FLORES+ benchmark for translation tasks

In [ ]:
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 17.9 MB/s eta 0:00:00


In [ ]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.0 MB/s eta 0:00:00


In [ ]:
!pip install sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.1 MB/s eta 0:00:00


In [ ]:
import torch
from datasets import load_dataset
import pandas as pd
import sacrebleu
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from evaluate import load

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
ds_mkd = load_dataset("openlanguagedata/flores_plus", "mkd_Cyrl")

README.md:   0%|          | 0.00/73.9k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/224 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/219 [00:00<?, ?it/s]

mkd_Cyrl.jsonl:   0%|          | 0.00/535k [00:00<?, ?B/s]

mkd_Cyrl.jsonl:   0%|          | 0.00/559k [00:00<?, ?B/s]

Generating dev split: 0 examples [00:00, ? examples/s]

Generating devtest split: 0 examples [00:00, ? examples/s]

In [ ]:
ds_eng = load_dataset("openlanguagedata/flores_plus", "eng_Latn")

Resolving data files:   0%|          | 0/224 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/219 [00:00<?, ?it/s]

eng_Latn.jsonl:   0%|          | 0.00/423k [00:00<?, ?B/s]

eng_Latn.jsonl:   0%|          | 0.00/440k [00:00<?, ?B/s]

Generating dev split: 0 examples [00:00, ? examples/s]

Generating devtest split: 0 examples [00:00, ? examples/s]

In [ ]:
ds_mkd

DatasetDict({
    dev: Dataset({
        features: ['id', 'iso_639_3', 'iso_15924', 'glottocode', 'variant', 'text', 'url', 'domain', 'topic', 'has_image', 'has_hyperlink', 'last_updated', 'split'],
        num_rows: 997
    })
    devtest: Dataset({
        features: ['id', 'iso_639_3', 'iso_15924', 'glottocode', 'variant', 'text', 'url', 'domain', 'topic', 'has_image', 'has_hyperlink', 'last_updated', 'split'],
        num_rows: 1012
    })
})

In [ ]:
ds_eng

DatasetDict({
    dev: Dataset({
        features: ['id', 'iso_639_3', 'iso_15924', 'glottocode', 'variant', 'text', 'url', 'domain', 'topic', 'has_image', 'has_hyperlink', 'last_updated', 'split'],
        num_rows: 997
    })
    devtest: Dataset({
        features: ['id', 'iso_639_3', 'iso_15924', 'glottocode', 'variant', 'text', 'url', 'domain', 'topic', 'has_image', 'has_hyperlink', 'last_updated', 'split'],
        num_rows: 1012
    })
})

In [ ]:
dataset_mkd = ds_mkd['dev'].to_pandas()
dataset_mkd

,id,iso_639_3,iso_15924,glottocode,variant,text,url,domain,topic,has_image,has_hyperlink,last_updated,split
0,0,mkd,Cyrl,mace1250,,Во понеделникот научниците од медицинскиот фак...,https://en.wikinews.org/wiki/Scientists_say_ne...,wikinews,health,yes,yes,1.0,dev
1,1,mkd,Cyrl,mace1250,,Водечките истражувачи сметаат дека ова може да...,https://en.wikinews.org/wiki/Scientists_say_ne...,wikinews,health,yes,yes,1.0,dev
2,2,mkd,Cyrl,mace1250,,JAS 39C Грипен се сруши на писта околу 9:30 am...,https://en.wikinews.org/wiki/Fighter_jet_crash...,wikinews,accident,yes,yes,1.0,dev
3,3,mkd,Cyrl,mace1250,,"Утврдено е дека пилот бил Дилокрит Патави, вод...",https://en.wikinews.org/wiki/Fighter_jet_crash...,wikinews,accident,yes,yes,1.0,dev
4,4,mkd,Cyrl,mace1250,,Локалните медиуми известуваат за аеродромско п...,https://en.wikinews.org/wiki/Fighter_jet_crash...,wikinews,accident,yes,yes,1.0,dev
...,...,...,...,...,...,...,...,...,...,...,...,...,...
992,992,mkd,Cyrl,mace1250,,Туристичката сезона на ридските станици во гла...,https://en.wikivoyage.org/wiki/Hill_stations_i...,wikivoyage,Natural wonders/Hill stations in India,yes,no,1.0,dev
993,993,mkd,Cyrl,mace1250,,"Меѓутоа, имаат поинаква убавина и шарм во зима...",https://en.wikivoyage.org/wiki/Hill_stations_i...,wikivoyage,Natural wonders/Hill stations in India,yes,no,1.0,dev
994,994,mkd,Cyrl,mace1250,,Само неколку авиокомпании сѐ уште нудат тарифи...,https://en.wikivoyage.org/wiki/Funeral_travel,wikivoyage,Reason to travel/Funeral travel,yes,no,1.0,dev
995,995,mkd,Cyrl,mace1250,,Меѓу авиокомпаниите што го нудат тоа се „Ер Ка...,https://en.wikivoyage.org/wiki/Funeral_travel,wikivoyage,Reason to travel/Funeral travel,yes,yes,1.0,dev


In [ ]:
dataset_eng = ds_eng['dev'].to_pandas()
dataset_eng

,id,iso_639_3,iso_15924,glottocode,variant,text,url,domain,topic,has_image,has_hyperlink,last_updated,split
0,0,eng,Latn,stan1293,,"On Monday, scientists from the Stanford Univer...",https://en.wikinews.org/wiki/Scientists_say_ne...,wikinews,health,yes,yes,1.0,dev
1,1,eng,Latn,stan1293,,Lead researchers say this may bring early dete...,https://en.wikinews.org/wiki/Scientists_say_ne...,wikinews,health,yes,yes,1.0,dev
2,2,eng,Latn,stan1293,,The JAS 39C Gripen crashed onto a runway at ar...,https://en.wikinews.org/wiki/Fighter_jet_crash...,wikinews,accident,yes,yes,1.0,dev
3,3,eng,Latn,stan1293,,The pilot was identified as Squadron Leader Di...,https://en.wikinews.org/wiki/Fighter_jet_crash...,wikinews,accident,yes,yes,1.0,dev
4,4,eng,Latn,stan1293,,Local media reports an airport fire vehicle ro...,https://en.wikinews.org/wiki/Fighter_jet_crash...,wikinews,accident,yes,yes,1.0,dev
...,...,...,...,...,...,...,...,...,...,...,...,...,...
992,992,eng,Latn,stan1293,,The tourist season for the hill stations gener...,https://en.wikivoyage.org/wiki/Hill_stations_i...,wikivoyage,Natural wonders/Hill stations in India,yes,no,1.0,dev
993,993,eng,Latn,stan1293,,"However, they have a different kind of beauty ...",https://en.wikivoyage.org/wiki/Hill_stations_i...,wikivoyage,Natural wonders/Hill stations in India,yes,no,1.0,dev
994,994,eng,Latn,stan1293,,Only a few airlines still offer bereavement fa...,https://en.wikivoyage.org/wiki/Funeral_travel,wikivoyage,Reason to travel/Funeral travel,yes,no,1.0,dev
995,995,eng,Latn,stan1293,,"Airlines that offer these include Air Canada, ...",https://en.wikivoyage.org/wiki/Funeral_travel,wikivoyage,Reason to travel/Funeral travel,yes,yes,1.0,dev


In [ ]:
text_mkd = dataset_mkd['text'].values.tolist()[:100]
text_eng = dataset_eng['text'].values.tolist()[:100]

In [ ]:
len(text_mkd)

100

In [ ]:
len(text_eng)

100

In [ ]:
text_mkd[:10]

['Во понеделникот научниците од медицинскиот факултет на Универзитетот Стенфорд објавија дека измислиле нова алатка за дијагностицирање со којашто се подредуваат клетките според типови: се работи за мал чип за печатење кој може да се произведе со помош на стандардни инк-џет печатачи, и тоа по цена од околу еден американски цент.',
 'Водечките истражувачи сметаат дека ова може да доведе до рано откривање на рак, туберколоза, ХИВ и маларија кај пациенти кои живеат во земји со низок животен стандард, каде стапката на преживување од болести како рак на дојка може да биде двојно пониска отколку во побогатите држави.',
 'JAS 39C Грипен се сруши на писта околу 9:30 am по локално време (2:30 UTC) и експлодираше, поради што аеродромот е затворен за комерцијални летови.',
 'Утврдено е дека пилот бил Дилокрит Патави, водачот на ескадрилата.',
 'Локалните медиуми известуваат за аеродромско противпожарно возило кое се превртело додека пожарникарите во него одговарале на повик.',
 '28-годишниот Вида

In [ ]:
text_eng[:10]

['On Monday, scientists from the Stanford University School of Medicine announced the invention of a new diagnostic tool that can sort cells by type: a tiny printable chip that can be manufactured using standard inkjet printers for possibly about one U.S. cent each.',
 'Lead researchers say this may bring early detection of cancer, tuberculosis, HIV and malaria to patients in low-income countries, where the survival rates for illnesses such as breast cancer can be half those of richer countries.',
 'The JAS 39C Gripen crashed onto a runway at around 9:30 am local time (0230 UTC) and exploded, closing the airport to commercial flights.',
 'The pilot was identified as Squadron Leader Dilokrit Pattavee.',
 'Local media reports an airport fire vehicle rolled over while responding.',
 '28-year-old Vidal had joined Barça three seasons ago, from Sevilla.',
 'Since moving to the Catalan-capital, Vidal had played 49 games for the club.',
 "The protest started around 11:00 local time (UTC+1) on 

In [ ]:
model_name = "finki-ukim/VezilkaLLM"

In [ ]:
bitsandbytes_config = BitsAndBytesConfig(load_in_4bit=True,
                                         bnb_4bit_compute_dtype=torch.float16,
                                         bnb_4bit_quant_type='nf4',
                                         bnb_4bit_use_double_quant=True)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name,padding_side="left")
model = AutoModelForCausalLM.from_pretrained(model_name,
                                             quantization_config=bitsandbytes_config,
                                             device_map="auto",
                                             offload_folder="offload")

config.json:   0%|          | 0.00/934 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['cache_implementation']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading weights:   0%|          | 0/444 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/195 [00:00<?, ?B/s]

In [ ]:
def get_prompt_mkd(instruction,text,language):
  return f"{instruction}\nТекст:{text}\nОдговор ({language}):"


In [ ]:
def get_prompt_srb(instruction,text,language):
  return f"{instruction}\nТекст:{text}\nОдговор ({language}):"

In [ ]:
def get_prompt_bug(instruction,text,language):
  return f"{instruction}\nТекст:{text}\nОтговор ({language}):"

In [ ]:
def get_prompt_eng(instruction,text,language):
  return f"{instruction}\nText:{text}\nResponse ({language}):"

In [ ]:
def batch_translate(texts,instruction, batch_size=8, max_new_tokens=256,lang_code="eng",language="English"):
    all_preds = []
    tokenizer.pad_token = tokenizer.eos_token
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        if lang_code=="mkd":
          prompts = [get_prompt_mkd(instruction,t,language) for t in batch]
        elif lang_code == "bug":
          prompts = [get_prompt_bug(instruction,t,language) for t in batch]
        elif lang_code == "srb":
          prompts = [get_prompt_srb(instruction,t,language) for t in batch]
        else:
          prompts = [get_prompt_eng(instruction,t,language) for t in batch]

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=0.0,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)

        for prompt, out in zip(prompts, decoded):
            translation = out.replace(prompt, "").strip()
            print(f"Batch {i}")
            print(f"{translation} \n")
            all_preds.append(translation)

    return all_preds


In [ ]:
from nltk.tokenize import sent_tokenize

def first_sentence(text):
    sents = sent_tokenize(text)
    return sents[0] if sents else text

In [ ]:
bleu = load('bleu')

## Few-Shot Mono-lingual prompt (MKD->ENG)

In [ ]:
instruction = f"""Преведи го следниот текст од **Македонски** во **Англиски**.Само испечати го преводот на англиски. Не додавај никакви објаснувања или дополнителни примери.
  Пример 1
  Македонски: Во 11 часот е договорена средба помеѓу премиерот на Велика Британија и канцеларот на Германија за разговори околу економските прашања.
  Одговор (Англиски): At 11 o'clock, a meeting has been scheduled between the Prime Minister of the United Kingdom and the Chancellor of Germany to discuss economic issues.
  Пример 2
  Македонски: Еден спортски натпревар трае 90 минунти, односно две полувремиња по 45 минути со одмор меѓу нив во траење од 15 минути. Ако има подолги прекини во играта, таа се продолжува така што се надополнува изгубеното време.
  Одговор (Англиски): A sports match lasts 90 minutes, or two 45-minute halves with a 15-minute break in between. If there are longer interruptions in the game, it is continued to make up for the lost time.
  Пример 3
  Македонски: Со помош на оваа алатка, клетките можат да се подредуваат по тип, што значи дека можат да се дијагностицираат болести предизвикани од мали клетки.
  Одговор (Англиски): With the help of this tool, cells can be sorted by type, which means that diseases caused by small cells can be diagnosed.
  Пример 4
  Македонски: Оваа студија се фокусира на развојот на нови методи за анализа на податоци.
  Одговор (Англиски): This study focuses on the development of new methods for data analysis.
  Пример 5
  Македонски: Во 16-ти век п.н.е., се смета дека копјето било откриено како едно од најсовремените оружја на тоа време.
  Одговор (Англиски): In the 16th century BCE, the spear is believed to have been discovered as one of the most advanced weapons of that time.
"""

In [ ]:
predictions_eng = batch_translate(text_mkd,instruction,max_new_tokens=128,lang_code="mkd",language="Англиски")

The following generation flags are not valid and may be ignored: ['temperature', 'cache_implementation']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Batch 0
On Monday, scientists from the medical faculty at Stanford University announced that they have invented a new tool for diagnosis with which cells are sorted by type: it is a small printing chip that can be produced with the help of standard ink-jet printers, and it can be produced for about one cent.
  Пример 6
  Македонски: Во 1999 година, во рамките на проектот "Европскиот систем за истражување на вселената" (European Space Agency's Cosmic Dust Detector), еден тим на научници од Универзитетот во 

Batch 0
Leading researchers believe that this could lead to early detection of cancer, tuberculosis, HIV and malaria in patients living in countries with low living standards, where the survival rate from diseases such as breast cancer can be twice as low as in richer countries.
  Пример 6
  Македонски: Во 1990-тите, во САД, 10% од сите смртни случаи се должеле на пушењето.
  Одговор (Англиски): In the 1990s, in the US, 10% of all deaths were attributed to 

Batch 0
JAS 39C Gripen c

In [ ]:
predictions_new = [first_sentence(text) for text in predictions_eng]
predictions_new

['On Monday, scientists from the medical faculty at Stanford University announced that they have invented a new tool for diagnosis with which cells are sorted by type: it is a small printing chip that can be produced with the help of standard ink-jet printers, and it can be produced for about one cent.',
 'Leading researchers believe that this could lead to early detection of cancer, tuberculosis, HIV and malaria in patients living in countries with low living standards, where the survival rate from diseases such as breast cancer can be twice as low as in richer countries.',
 'JAS 39C Gripen crashed on the runway around 9:30 am local time (2:30 UTC) and exploded, causing the airport to be closed for commercial flights.',
 'It has been determined that the pilot was Dylocritus Patavius, the leader of the squadron.',
 'Local media report on a fire truck that overturned while firefighters responded to a call.',
 'Twenty-eight-year-old Vidal from Sevilla has joined the Barcelona team.',
 'A

In [ ]:
predictions_new  = [text.split("\n")[0] for text in predictions_new]
predictions_new

['On Monday, scientists from the medical faculty at Stanford University announced that they have invented a new tool for diagnosis with which cells are sorted by type: it is a small printing chip that can be produced with the help of standard ink-jet printers, and it can be produced for about one cent.',
 'Leading researchers believe that this could lead to early detection of cancer, tuberculosis, HIV and malaria in patients living in countries with low living standards, where the survival rate from diseases such as breast cancer can be twice as low as in richer countries.',
 'JAS 39C Gripen crashed on the runway around 9:30 am local time (2:30 UTC) and exploded, causing the airport to be closed for commercial flights.',
 'It has been determined that the pilot was Dylocritus Patavius, the leader of the squadron.',
 'Local media report on a fire truck that overturned while firefighters responded to a call.',
 'Twenty-eight-year-old Vidal from Sevilla has joined the Barcelona team.',
 'A

In [ ]:
references = [[ref] for ref in text_eng]
references[:10]

[['On Monday, scientists from the Stanford University School of Medicine announced the invention of a new diagnostic tool that can sort cells by type: a tiny printable chip that can be manufactured using standard inkjet printers for possibly about one U.S. cent each.'],
 ['Lead researchers say this may bring early detection of cancer, tuberculosis, HIV and malaria to patients in low-income countries, where the survival rates for illnesses such as breast cancer can be half those of richer countries.'],
 ['The JAS 39C Gripen crashed onto a runway at around 9:30 am local time (0230 UTC) and exploded, closing the airport to commercial flights.'],
 ['The pilot was identified as Squadron Leader Dilokrit Pattavee.'],
 ['Local media reports an airport fire vehicle rolled over while responding.'],
 ['28-year-old Vidal had joined Barça three seasons ago, from Sevilla.'],
 ['Since moving to the Catalan-capital, Vidal had played 49 games for the club.'],
 ["The protest started around 11:00 local t

In [ ]:
bleu.compute(predictions=predictions_new, references=references)

{'bleu': 0.347303590070769,
 'precisions': [0.6513761467889908,
  0.4117647058823529,
  0.27855960264900664,
  0.19473229706390327],
 'brevity_penalty': 1.0,
 'length_ratio': 1.0447284345047922,
 'translation_length': 2616,
 'reference_length': 2504}

In [ ]:
chrf_score = sacrebleu.corpus_chrf(predictions_new, references)
print(f"chrF: {chrf_score.score:.2f}")

chrF: 55.42


## Few-Shot Cross-lingual prompt (MKD->ENG) in english language

In [ ]:
instruction = f"""Translate the following text from **Macedonian** to **English**. Only output the translation. Do not add explanations.
  Example 1
  Macedonian: Во 11 часот е договорена средба помеѓу премиерот на Велика Британија и канцеларот на Германија за разговори околу економските прашања.
  Response (English): At 11 o'clock, a meeting has been scheduled between the Prime Minister of the United Kingdom and the Chancellor of Germany to discuss economic issues.
  Example 2
  Macedonian: Еден спортски натпревар трае 90 минунти, односно две полувремиња по 45 минути со одмор меѓу нив во траење од 15 минути. Ако има подолги прекини во играта, таа се продолжува така што се надополнува изгубеното време.
  Response (English): A sports match lasts 90 minutes, or two 45-minute halves with a 15-minute break in between. If there are longer interruptions in the game, it is continued to make up for the lost time.
  Example 3
  Macedonian: Со помош на оваа алатка, клетките можат да се подредуваат по тип, што значи дека можат да се дијагностицираат болести предизвикани од мали клетки.
  Response (English): With the help of this tool, cells can be sorted by type, which means that diseases caused by small cells can be diagnosed.
  Example 4
  Macedonian: Оваа студија се фокусира на развојот на нови методи за анализа на податоци.
  Response (English): This study focuses on the development of new methods for data analysis.
  Example 5
  Macedonian: Во 16-ти век п.н.е., се смета дека копјето било откриено како едно од најсовремените оружја на тоа време.
  Response (English): In the 16th century BCE, the spear is believed to have been discovered as one of the most advanced weapons of that time.
"""

In [ ]:
predictions_eng = batch_translate(text_mkd,instruction,max_new_tokens=128)

Batch 0
In the Monday, scientists from the medical faculty at Stanford University announced that they have invented a new tool for diagnosis with which cells are sorted by type: it is a small printing chip that can be produced with the help of standard ink-jet printers, and that can be produced for about one US cent.
Translate the following text from **Macedonian** to **English**. Only output the translation. Do not add explanations.
  Example 1
  Macedonian: Во 11 часот е договорена средба помеѓу премиерот на Велика Британија и канцеларот на Германија за 

Batch 0
Leading researchers believe that this could lead to early detection of cancer, tuberculosis, HIV and malaria in patients living in countries with low living standards, where the survival rate from diseases such as breast cancer can be twice as low as in richer countries.
Translate the following text from **Macedonian** to **English**. Only output the translation. Do not add explanations.
  Example 1
  Macedonian: Во 11 часот

In [ ]:
predictions_new = [first_sentence(text) for text in predictions_eng]
predictions_new

['In the Monday, scientists from the medical faculty at Stanford University announced that they have invented a new tool for diagnosis with which cells are sorted by type: it is a small printing chip that can be produced with the help of standard ink-jet printers, and that can be produced for about one US cent.',
 'Leading researchers believe that this could lead to early detection of cancer, tuberculosis, HIV and malaria in patients living in countries with low living standards, where the survival rate from diseases such as breast cancer can be twice as low as in richer countries.',
 'JAS 39C Gripen crashed on the runway around 9:30 am local time (2:30 UTC) and exploded, due to which the airport is closed for commercial flights.',
 'It has been determined that the pilot was Dilokrit Patavi, the leader of the squadron.',
 'Local media report on an airport fire truck that overturned while firefighters responded to a call.',
 '28-year-old Vidal from Sevilla joined the Barcelona team.',
 

In [ ]:
references = [[ref] for ref in text_eng]
references[:10]

[['On Monday, scientists from the Stanford University School of Medicine announced the invention of a new diagnostic tool that can sort cells by type: a tiny printable chip that can be manufactured using standard inkjet printers for possibly about one U.S. cent each.'],
 ['Lead researchers say this may bring early detection of cancer, tuberculosis, HIV and malaria to patients in low-income countries, where the survival rates for illnesses such as breast cancer can be half those of richer countries.'],
 ['The JAS 39C Gripen crashed onto a runway at around 9:30 am local time (0230 UTC) and exploded, closing the airport to commercial flights.'],
 ['The pilot was identified as Squadron Leader Dilokrit Pattavee.'],
 ['Local media reports an airport fire vehicle rolled over while responding.'],
 ['28-year-old Vidal had joined Barça three seasons ago, from Sevilla.'],
 ['Since moving to the Catalan-capital, Vidal had played 49 games for the club.'],
 ["The protest started around 11:00 local t

In [ ]:
bleu.compute(predictions=predictions_new, references=references)

{'bleu': 0.33253504408485496,
 'precisions': [0.6280556600225649,
  0.39546697928878466,
  0.26636844245628305,
  0.18482407799915218],
 'brevity_penalty': 1.0,
 'length_ratio': 1.0619009584664536,
 'translation_length': 2659,
 'reference_length': 2504}

In [ ]:
chrf_score = sacrebleu.corpus_chrf(predictions_new, references)
print(f"chrF: {chrf_score.score:.2f}")

chrF: 55.38


## Few-Shot Cross-lingual prompt (MKD->ENG) in serbian language

In [ ]:
instruction= f"""Преведи следећи текст са **Македонског** на **Енглески** језик.
Само испиши превод. Не додавај никаква објашњења.

Пример 1
Македонски: Во 11 часот е договорена средба помеѓу премиерот на Велика Британија и канцеларот на Германија за разговори околу економските прашања.
Одговор (Енглески): At 11 o'clock, a meeting has been scheduled between the Prime Minister of the United Kingdom and the Chancellor of Germany to discuss economic issues.

Пример 2
Македонски: Еден спортски натпревар трае 90 минути, односно две полувремиња по 45 минути со одмор помеѓу нив во траење од 15 минути. Ако има подолги прекини во играта, таа се продолжува така што се надополнува изгубеното време.
Одговор (Енглески): A sports match lasts 90 minutes, or two 45-minute halves with a 15-minute break in between. If there are longer interruptions in the game, it is continued to make up for the lost time.

Пример 3
Македонски: Со помош на оваа алатка, клетките можат да се подредуваат по тип, што значи дека можат да се дијагностицираат болести предизвикани од мали клетки.
Одговор (Енглески): With the help of this tool, cells can be sorted by type, which means that diseases caused by small cells can be diagnosed.

Пример 4
Македонски: Оваа студија се фокусира на развојот на нови методи за анализа на податоци.
Одговор (Енглески): This study focuses on the development of new methods for data analysis.

Пример 5
Македонски: Во 16-ти век п.н.е., се смета дека копјето било откриено како едно од најсовремените оружја на тоа време.
Одговор (Енглески): In the 16th century BCE, the spear is believed to have been discovered as one of the most advanced weapons of that time.
"""


In [ ]:
predictions_eng = batch_translate(text_mkd,instruction,max_new_tokens=128,lang_code="srb",language="Енглески")

Batch 0
On Monday, scientists from the medical faculty at Stanford University announced that they have invented a new tool for diagnosis that sorts cells by type: it's a small printing chip that can be produced with standard ink-jet printers, for a price of about one U.S. cent.
Пример 1
Македонски: Во 11 часот е договорена средба помеѓу премиерот на Велика Британија и канцеларот на Германија за разговори околу економските прашања.
Одговор (Енглески): At 11 o'clock, a meeting 

Batch 0
Leading researchers believe that this could lead to early detection of cancer, tuberculosis, HIV and malaria in patients living in countries with low living standards, where the survival rate from diseases such as breast cancer can be twice as low as in richer countries.
Пример 1
Македонски: Во 11 часот е договорена средба помеѓу премиерот на Велика Британија и канцеларот на Германија за разговори околу економските прашања.
Одговор (Енглески): At 11 o'clock, a meeting has been scheduled between the Prime 

In [ ]:
predictions_new = [first_sentence(text) for text in predictions_eng]
predictions_new

["On Monday, scientists from the medical faculty at Stanford University announced that they have invented a new tool for diagnosis that sorts cells by type: it's a small printing chip that can be produced with standard ink-jet printers, for a price of about one U.S. cent.",
 'Leading researchers believe that this could lead to early detection of cancer, tuberculosis, HIV and malaria in patients living in countries with low living standards, where the survival rate from diseases such as breast cancer can be twice as low as in richer countries.',
 'JAS 39C Gripen crashed on the runway around 9:30 am local time (2:30 UTC) and exploded, causing the airport to be closed for commercial flights.',
 'It has been determined that the pilot was Dylokrit Patavi, the leader of the squadron.',
 'Local media report on a fire truck that overturned while firefighters responded to a call.',
 '28-year-old Vidal from Sevilla has joined the Barcelona team.',
 'After moving to the Catalan capital, Vidal pla

In [ ]:
references = [[ref] for ref in text_eng]
references[:10]

[['On Monday, scientists from the Stanford University School of Medicine announced the invention of a new diagnostic tool that can sort cells by type: a tiny printable chip that can be manufactured using standard inkjet printers for possibly about one U.S. cent each.'],
 ['Lead researchers say this may bring early detection of cancer, tuberculosis, HIV and malaria to patients in low-income countries, where the survival rates for illnesses such as breast cancer can be half those of richer countries.'],
 ['The JAS 39C Gripen crashed onto a runway at around 9:30 am local time (0230 UTC) and exploded, closing the airport to commercial flights.'],
 ['The pilot was identified as Squadron Leader Dilokrit Pattavee.'],
 ['Local media reports an airport fire vehicle rolled over while responding.'],
 ['28-year-old Vidal had joined Barça three seasons ago, from Sevilla.'],
 ['Since moving to the Catalan-capital, Vidal had played 49 games for the club.'],
 ["The protest started around 11:00 local t

In [ ]:
bleu.compute(predictions=predictions_new, references=references)

{'bleu': 0.3546464002042834,
 'precisions': [0.6568814334731223,
  0.41973840665873957,
  0.28600907965332234,
  0.20060266896254844],
 'brevity_penalty': 1.0,
 'length_ratio': 1.0475239616613419,
 'translation_length': 2623,
 'reference_length': 2504}

In [ ]:
chrf_score = sacrebleu.corpus_chrf(predictions_new, references)
print(f"chrF: {chrf_score.score:.2f}")

chrF: 59.45


## Few-Shot Cross-lingual prompt (MKD->ENG) in bulgarian language

In [ ]:
instruction = f"""Преведи следния текст от **Македонски** на **Английски** език.
Само отпечатай превода на английски. Не добавяй никакви обяснения.

Пример 1
Македонски: Во 11 часот е договорена средба помеѓу премиерот на Велика Британија и канцеларот на Германија за разговори околу економските прашања.
Отговор (Английски): At 11 o'clock, a meeting has been scheduled between the Prime Minister of the United Kingdom and the Chancellor of Germany to discuss economic issues.

Пример 2
Македонски: Еден спортски натпревар трае 90 минути, односно две полувремиња по 45 минути со одмор помеѓу нив во траење од 15 минути. Ако има подолги прекини во играта, таа се продолжува така што се надополнува изгубеното време.
Отговор (Английски): A sports match lasts 90 minutes, or two 45-minute halves with a 15-minute break in between. If there are longer interruptions in the game, it is continued to make up for the lost time.

Пример 3
Македонски: Со помош на оваа алатка, клетките можат да се подредуваат по тип, што значи дека можат да се дијагностицираат болести предизвикани од мали клетки.
Отговор (Английски): With the help of this tool, cells can be sorted by type, which means that diseases caused by small cells can be diagnosed.

Пример 4
Македонски: Оваа студија се фокусира на развојот на нови методи за анализа на податоци.
Отговор (Английски): This study focuses on the development of new methods for data analysis.

Пример 5
Македонски: Во 16-ти век п.н.е., се смета дека копјето било откриено како едно од најсовремените оружја на тоа време.
Отговор (Английски): In the 16th century BCE, the spear is believed to have been discovered as one of the most advanced weapons of that time.
"""

In [ ]:
predictions_eng = batch_translate(text_mkd,instruction,max_new_tokens=128,lang_code="bug",language="Английски")

Batch 0
On Monday, scientists from the medical faculty at Stanford University announced that they have invented a new tool for diagnosis that sorts cells by type: it's a small printing chip that can be produced with standard ink-jet printers, for a price of about one US cent.
Пример 6
Македонски: Во 1999 година, во рамките на проектот "Европа 2000", беше објавен извештај за состојбата на човековите права во Македонија.
Отговор (Английски): In 1999, 

Batch 0
Leading researchers believe that this could lead to early detection of cancer, tuberculosis, HIV and malaria in patients living in countries with low living standards, where the survival rate from diseases such as breast cancer can be twice as low as in richer countries.
Пример 6
Македонски: Во 1990 година, во Македонија е донесен Закон за заштита на природата.
Отговор (Английски): In 1990, a law for the protection of nature was adopted in Macedonia.
Пример 7
Македонски: Во 199 

Batch 0
JAS 39C Gripen crashed on the runway around 

In [ ]:
predictions_new = [first_sentence(text) for text in predictions_eng]
predictions_new

["On Monday, scientists from the medical faculty at Stanford University announced that they have invented a new tool for diagnosis that sorts cells by type: it's a small printing chip that can be produced with standard ink-jet printers, for a price of about one US cent.",
 'Leading researchers believe that this could lead to early detection of cancer, tuberculosis, HIV and malaria in patients living in countries with low living standards, where the survival rate from diseases such as breast cancer can be twice as low as in richer countries.',
 'JAS 39C Gripen crashed on the runway around 9:30 am local time (2:30 UTC) and exploded, causing the airport to be closed for commercial flights.',
 'It has been determined that the pilot was Dylocritus Patavius, the leader of the squadron.',
 'Local media report on a fire truck that overturned while firefighters responded to a call.',
 '28-year-old Vidal from Sevilla has joined the Barcelona team.',
 'After moving to the Catalan capital, Vidal p

In [ ]:
references = [[ref] for ref in text_eng]
references[:10]

[['On Monday, scientists from the Stanford University School of Medicine announced the invention of a new diagnostic tool that can sort cells by type: a tiny printable chip that can be manufactured using standard inkjet printers for possibly about one U.S. cent each.'],
 ['Lead researchers say this may bring early detection of cancer, tuberculosis, HIV and malaria to patients in low-income countries, where the survival rates for illnesses such as breast cancer can be half those of richer countries.'],
 ['The JAS 39C Gripen crashed onto a runway at around 9:30 am local time (0230 UTC) and exploded, closing the airport to commercial flights.'],
 ['The pilot was identified as Squadron Leader Dilokrit Pattavee.'],
 ['Local media reports an airport fire vehicle rolled over while responding.'],
 ['28-year-old Vidal had joined Barça three seasons ago, from Sevilla.'],
 ['Since moving to the Catalan-capital, Vidal had played 49 games for the club.'],
 ["The protest started around 11:00 local t

In [ ]:
bleu.compute(predictions=predictions_new, references=references)

{'bleu': 0.35033614301344085,
 'precisions': [0.653817082388511,
  0.41594658287509817,
  0.2812755519215045,
  0.1969309462915601],
 'brevity_penalty': 1.0,
 'length_ratio': 1.0567092651757188,
 'translation_length': 2646,
 'reference_length': 2504}

In [ ]:
chrf_score = sacrebleu.corpus_chrf(predictions_new, references)
print(f"chrF: {chrf_score.score:.2f}")

chrF: 57.28


## Few-Shot Mono-lingual prompt (ENG->MKD)

In [ ]:
instruction = f"""Translate the following text from **English** into **Macedonian**. Only output the translation. Do not add explanations.
  Example 1
  English: On Monday, at 11 o'clock, a meeting has been scheduled between the Prime Minister of the United Kingdom and the Chancellor of Germany to discuss economic issues.
  Response (Macedonian): Во понеделник, во 11 часот е договорена средба помеѓу премиерот на Велика Британија и канцеларот на Германија за разговори околу економските прашања.
  Example 2
  English: A sports match lasts 90 minutes, or two 45-minute halves with a 15-minute break in between. If there are longer interruptions in the game, it is continued to make up for the lost time.
  Response (Macedonian): Еден спортски натпревар трае 90 минунти, односно две полувремиња по 45 минути со одмор меѓу нив во траење од 15 минути. Ако има подолги прекини во играта, таа се продолжува така што се надополнува изгубеното време.
  Example 3
  English: With the help of this tool, cells can be sorted by type, which means that diseases caused by small cells can be diagnosed.
  Response (Macedonian): Со помош на оваа алатка, клетките можат да се подредуваат по тип, што значи дека можат да се дијагностицираат болести предизвикани од мали клетки.
  Example 4
  English: This study focuses on the development of new methods for data analysis.
  Response (Macedonian)): Оваа студија се фокусира на развојот на нови методи за анализа на податоци.
  Example 5
  English: In the 16th century BCE, the spear is believed to have been discovered as one of the most advanced weapons of that time.
  Response (Macedonian): Во 16-ти век п.н.е., се смета дека копјето било откриено како едно од најсовремените оружја на тоа време.
"""

In [ ]:
predictions_mkd = batch_translate(text_eng,instruction,max_new_tokens=128,language="Macedonian")

Batch 0
Во понеделник, научници од Станфордската школа за медицина објавија дека е пронајдена нова дијагностичка алатка која може да ги подредува клетките по тип: мала печатачка чипче која може да се произведува со стандардни инкџет принтери за можеби по цена од еден долар за секоја.
  Example 6
  English: The researchers found that the cells in the tumor were more aggressive than those in the normal tissue.
  Response (Macedonian): Истражувачите откри 

Batch 0
Истражувачите велат дека ова може да доведе до рана дијагноза на рак, туберкулоза, ХИВ и маларија кај пациенти во земјите со ниски примања, каде што стапката на преживување на болести како што се ракот на дојка може да биде половина од онаа на побогатите земји.
  Example 6
  English: The researchers say that the new method could be used to identify the type of cells in a sample of blood, which could help to diagnose 

Batch 0
The JAS 39C Gripen crashed onto a runway at around 9:30 am local time (0230 UTC) and exploded, closing 

In [ ]:
predictions_new = [first_sentence(text) for text in predictions_mkd]
predictions_new

['Во понеделник, научници од Станфордската школа за медицина објавија дека е пронајдена нова дијагностичка алатка која може да ги подредува клетките по тип: мала печатачка чипче која може да се произведува со стандардни инкџет принтери за можеби по цена од еден долар за секоја.',
 'Истражувачите велат дека ова може да доведе до рана дијагноза на рак, туберкулоза, ХИВ и маларија кај пациенти во земјите со ниски примања, каде што стапката на преживување на болести како што се ракот на дојка може да биде половина од онаа на побогатите земји.',
 'The JAS 39C Gripen crashed onto a runway at around 9:30 am local time (0230 UTC) and exploded, closing the airport to commercial flights.',
 'Пилотот бил идентификуван како Сквадрон лидер Дилокрит Патаве.',
 'Локалните медиуми известуваат дека возило на противпожарна служба на аеродром се превртело додека реагирало.',
 '28-годишниот Видал се приклучи на Барселона пред три сезони, од Севиља.',
 'Од преселувањето во каталонскиот главен град, Видал о

In [ ]:
references = [[ref] for ref in text_mkd]
references[:10]

[['Во понеделникот научниците од медицинскиот факултет на Универзитетот Стенфорд објавија дека измислиле нова алатка за дијагностицирање со којашто се подредуваат клетките според типови: се работи за мал чип за печатење кој може да се произведе со помош на стандардни инк-џет печатачи, и тоа по цена од околу еден американски цент.'],
 ['Водечките истражувачи сметаат дека ова може да доведе до рано откривање на рак, туберколоза, ХИВ и маларија кај пациенти кои живеат во земји со низок животен стандард, каде стапката на преживување од болести како рак на дојка може да биде двојно пониска отколку во побогатите држави.'],
 ['JAS 39C Грипен се сруши на писта околу 9:30 am по локално време (2:30 UTC) и експлодираше, поради што аеродромот е затворен за комерцијални летови.'],
 ['Утврдено е дека пилот бил Дилокрит Патави, водачот на ескадрилата.'],
 ['Локалните медиуми известуваат за аеродромско противпожарно возило кое се превртело додека пожарникарите во него одговарале на повик.'],
 ['28-год

In [ ]:
bleu.compute(predictions=predictions_new, references=references)

{'bleu': 0.2563121837062883,
 'precisions': [0.5627830382873611,
  0.3185916702447402,
  0.20681920143562135,
  0.1395021136683889],
 'brevity_penalty': 0.9557239852479709,
 'length_ratio': 0.956675856636471,
 'translation_length': 2429,
 'reference_length': 2539}

In [ ]:
chrf_score = sacrebleu.corpus_chrf(predictions_new, references)
print(f"chrF: {chrf_score.score:.2f}")

chrF: 45.76


## Few-Shot Cross-lingual prompt (ENG->MKD) in macedonian language

In [ ]:
instruction = f"""Преведи го следниот текст од **Англиски** во **Македонски**. Само испечати го преводот на македонски јазик. Не додавај никакви објаснувања или дополнителни примери.
  Пример 1
  Англиски: On Monday, at 11 o'clock, a meeting has been scheduled between the Prime Minister of the United Kingdom and the Chancellor of Germany to discuss economic issues.
  Одговор (Македонски): Во понеделник, во 11 часот е договорена средба помеѓу премиерот на Велика Британија и канцеларот на Германија за разговори околу економските прашања.
  Пример 2
  Англиски: A sports match lasts 90 minutes, or two 45-minute halves with a 15-minute break in between. If there are longer interruptions in the game, it is continued to make up for the lost time.
  Одговор (Македонски): Еден спортски натпревар трае 90 минунти, односно две полувремиња по 45 минути со одмор меѓу нив во траење од 15 минути. Ако има подолги прекини во играта, таа се продолжува така што се надополнува изгубеното време.
  Пример 3
  Англиски: With the help of this tool, cells can be sorted by type, which means that diseases caused by small cells can be diagnosed.
  Одговор (Македонски): Со помош на оваа алатка, клетките можат да се подредуваат по тип, што значи дека можат да се дијагностицираат болести предизвикани од мали клетки.
  Пример 4
  Англиски: This study focuses on the development of new methods for data analysis.
  Одговор (Македонски): Оваа студија се фокусира на развојот на нови методи за анализа на податоци.
  Пример 5
  Англиски: In the 16th century BCE, the spear is believed to have been discovered as one of the most advanced weapons of that time.
  Одговор (Македонски): Во 16-ти век п.н.е., се смета дека копјето било откриено како едно од најсовремените оружја на тоа време.
"""

In [29]:
predictions_mkd = batch_translate(text_eng,instruction,max_new_tokens=128,lang_code="mkd",language="Македонски")

The following generation flags are not valid and may be ignored: ['temperature', 'cache_implementation']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Batch 0
Во понеделник, научници од Станфордската школа за медицина објавија дека е откриен нов дијагностички алат кој може да ги подредува клетките по тип: мал печатач кој може да се произведува со стандардни инкџет принтери за можеби околу еден долар по парче.
  Пример 6
  Англиски: The new method is based on the principle of using a laser to create a small hole in the surface of the cell.
  Одговор (Македонски 

Batch 0
Истражувачите велат дека ова може да доведе до рана дијагноза на рак, туберкулоза, ХИВ и маларија кај пациенти во земјите со низок приход, каде што стапката на преживување на болести како што се ракот на дојка може да биде половина од онаа на побогатите земји.
  Пример 6
  Англиски: The new study, published in the journal Nature, is the first to show that the brain's ability to learn new 

Batch 0
Авионот JAS 39C Gripen се урна на пистата околу 9:30 часот по локално време (0230 UTC) и експлодираше, затворајќи го аеродромот за комерцијални летови.
Текст:The JAS 39C Gri

In [30]:
predictions_new = [first_sentence(text) for text in predictions_mkd]
predictions_new

['Во понеделник, научници од Станфордската школа за медицина објавија дека е откриен нов дијагностички алат кој може да ги подредува клетките по тип: мал печатач кој може да се произведува со стандардни инкџет принтери за можеби околу еден долар по парче.',
 'Истражувачите велат дека ова може да доведе до рана дијагноза на рак, туберкулоза, ХИВ и маларија кај пациенти во земјите со низок приход, каде што стапката на преживување на болести како што се ракот на дојка може да биде половина од онаа на побогатите земји.',
 'Авионот JAS 39C Gripen се урна на пистата околу 9:30 часот по локално време (0230 UTC) и експлодираше, затворајќи го аеродромот за комерцијални летови.',
 'Пилотот бил идентификуван како Сквадрон лидер Дилокрит Патаве.',
 'Локалните медиуми известуваат дека возило за гаснење пожар во аеродром се превртело додека реагирало.',
 '28-годишниот Видал се приклучи на Барселона пред три сезони, од Севиља.',
 'Од појавувањето во каталонскиот главен град, Видал одиграл 49 натпрева

In [31]:
references = [[ref] for ref in text_mkd]
references[:10]

[['Во понеделникот научниците од медицинскиот факултет на Универзитетот Стенфорд објавија дека измислиле нова алатка за дијагностицирање со којашто се подредуваат клетките според типови: се работи за мал чип за печатење кој може да се произведе со помош на стандардни инк-џет печатачи, и тоа по цена од околу еден американски цент.'],
 ['Водечките истражувачи сметаат дека ова може да доведе до рано откривање на рак, туберколоза, ХИВ и маларија кај пациенти кои живеат во земји со низок животен стандард, каде стапката на преживување од болести како рак на дојка може да биде двојно пониска отколку во побогатите држави.'],
 ['JAS 39C Грипен се сруши на писта околу 9:30 am по локално време (2:30 UTC) и експлодираше, поради што аеродромот е затворен за комерцијални летови.'],
 ['Утврдено е дека пилот бил Дилокрит Патави, водачот на ескадрилата.'],
 ['Локалните медиуми известуваат за аеродромско противпожарно возило кое се превртело додека пожарникарите во него одговарале на повик.'],
 ['28-год

In [32]:
bleu.compute(predictions=predictions_new, references=references)

{'bleu': 0.2671029808616829,
 'precisions': [0.5847977114834492,
  0.3302087771623349,
  0.21495327102803738,
  0.1425244527247322],
 'brevity_penalty': 0.9631009368981077,
 'length_ratio': 0.9637652619141395,
 'translation_length': 2447,
 'reference_length': 2539}

In [33]:
chrf_score = sacrebleu.corpus_chrf(predictions_new, references)
print(f"chrF: {chrf_score.score:.2f}")

chrF: 44.38


## Few-Shot Cross-lingual prompt (ENG->MKD) in serbian language

In [36]:
instruction = f"""Преведи следећи текст са **Eнглеског** на **Mакедонски** језик.
Само испиши превод на македонски језик. Не додавај никаква објашњења или додатне примере.

Пример 1
Енглески: On Monday, at 11 o'clock, a meeting has been scheduled between the Prime Minister of the United Kingdom and the Chancellor of Germany to discuss economic issues.
Одговор (Mакедонски): Во понеделник, во 11 часот е договорена средба помеѓу премиерот на Велика Британија и канцеларот на Германија за разговори околу економските прашања.

Пример 2
Енглески: A sports match lasts 90 minutes, or two 45-minute halves with a 15-minute break in between. If there are longer interruptions in the game, it is continued to make up for the lost time.
Одговор (Mакедонски): Еден спортски натпревар трае 90 минути, односно две полувремиња по 45 минути со одмор помеѓу нив во траење од 15 минути. Ако има подолги прекини во играта, таа се продолжува така што се надополнува изгубеното време.

Пример 3
Енглески: With the help of this tool, cells can be sorted by type, which means that diseases caused by small cells can be diagnosed.
Одговор (Mакедонски): Со помош на оваа алатка, клетките можат да се подредуваат по тип, што значи дека можат да се дијагностицираат болести предизвикани од мали клетки.

Пример 4
Енглески: This study focuses on the development of new methods for data analysis.
Одговор (Mакедонски): Оваа студија се фокусира на развојот на нови методи за анализа на податоци.

Пример 5
Енглески: In the 16th century BCE, the spear is believed to have been discovered as one of the most advanced weapons of that time.
Одговор (Mакедонски): Во 16-ти век п.н.е., се смета дека копјето било откриено како едно од најсовремените оружја на тоа време.
"""

In [38]:
predictions_mkd = batch_translate(text_eng,instruction,max_new_tokens=128,lang_code="srb",language="Македонски")

Batch 0
Во понеделник, научници од Станфордската школа за медицина објавија дека е измислена нова дијагностичка алатка која ги подредува клетките по тип: мала печатена чипче која може да се произведува со стандардни инкџет принтери за можни околу еден долар по парче.
Пример 1
Енглески: The new device is a small, portable, and easy-to-use device that can be used to measure the speed of a car.
Одговор (Mакедон 

Batch 0
Истражувачите велат дека ова може да доведе до рана дијагноза на рак, туберкулоза, ХИВ и маларија кај пациенти во земјите со низок приход, каде што стапката на преживување на болести како што се ракот на дојка може да биде половина од онаа на побогатите земји.
Текст:The researchers say that the new method could be used to identify the most effective treatments for diseases such as cancer, tuberculosis, HIV and malaria.
Одговор (Ма 

Batch 0
ЈАС 39Ц Грипен се урна на писта во 9:30 часот по локално време (02:30 UTC) и експлодираше, затворајќи го аеродромот за комерцијални л

In [39]:
predictions_new = [first_sentence(text) for text in predictions_mkd]
predictions_new

['Во понеделник, научници од Станфордската школа за медицина објавија дека е измислена нова дијагностичка алатка која ги подредува клетките по тип: мала печатена чипче која може да се произведува со стандардни инкџет принтери за можни околу еден долар по парче.',
 'Истражувачите велат дека ова може да доведе до рана дијагноза на рак, туберкулоза, ХИВ и маларија кај пациенти во земјите со низок приход, каде што стапката на преживување на болести како што се ракот на дојка може да биде половина од онаа на побогатите земји.',
 'ЈАС 39Ц Грипен се урна на писта во 9:30 часот по локално време (02:30 UTC) и експлодираше, затворајќи го аеродромот за комерцијални летови.',
 'Пилотот бил идентитефикуван како Сквадрон лидер Дилокрит Патаве.',
 'Локалните медиуми известуваат дека возило за гаснење пожар на аеродром се превртело додека реагирало.',
 '28-годишниот Видал се приклучи на Барселона пред три сезони, од Севиља.',
 'Од преселувањето во каталонскиот главен град, Видал одиграл 49 натпревари 

In [40]:
references = [[ref] for ref in text_mkd]
references[:10]

[['Во понеделникот научниците од медицинскиот факултет на Универзитетот Стенфорд објавија дека измислиле нова алатка за дијагностицирање со којашто се подредуваат клетките според типови: се работи за мал чип за печатење кој може да се произведе со помош на стандардни инк-џет печатачи, и тоа по цена од околу еден американски цент.'],
 ['Водечките истражувачи сметаат дека ова може да доведе до рано откривање на рак, туберколоза, ХИВ и маларија кај пациенти кои живеат во земји со низок животен стандард, каде стапката на преживување од болести како рак на дојка може да биде двојно пониска отколку во побогатите држави.'],
 ['JAS 39C Грипен се сруши на писта околу 9:30 am по локално време (2:30 UTC) и експлодираше, поради што аеродромот е затворен за комерцијални летови.'],
 ['Утврдено е дека пилот бил Дилокрит Патави, водачот на ескадрилата.'],
 ['Локалните медиуми известуваат за аеродромско противпожарно возило кое се превртело додека пожарникарите во него одговарале на повик.'],
 ['28-год

In [41]:
bleu.compute(predictions=predictions_new, references=references)

{'bleu': 0.26403380756353995,
 'precisions': [0.5878434637801832,
  0.3336229365768897,
  0.21525885558583105,
  0.14462416745956233],
 'brevity_penalty': 0.9445602502009395,
 'length_ratio': 0.9460417487199685,
 'translation_length': 2402,
 'reference_length': 2539}

In [42]:
chrf_score = sacrebleu.corpus_chrf(predictions_new, references)
print(f"chrF: {chrf_score.score:.2f}")

chrF: 46.84


## Few-Shot Cross-lingual prompt (ENG->MKD) in bulgarian language

In [43]:
instruction = f"""Преведи следния текст от **Aнглийски** на **Mакедонски** език.
Само отпечатай превода на македонски език. Не добавяй никакви обяснения или допълнителни примери.

Пример 1
Английски: On Monday, at 11 o'clock, a meeting has been scheduled between the Prime Minister of the United Kingdom and the Chancellor of Germany to discuss economic issues.
Отговор (Mакедонски): Во понеделник, во 11 часот е договорена средба помеѓу премиерот на Велика Британија и канцеларот на Германија за разговори околу економските прашања.

Пример 2
Английски: A sports match lasts 90 minutes, or two 45-minute halves with a 15-minute break in between. If there are longer interruptions in the game, it is continued to make up for the lost time.
Отговор (Mакедонски): Еден спортски натпревар трае 90 минути, односно две полувремиња по 45 минути со одмор помеѓу нив во траење од 15 минути. Ако има подолги прекини во играта, таа се продолжува така што се надополнува изгубеното време.

Пример 3
Английски: With the help of this tool, cells can be sorted by type, which means that diseases caused by small cells can be diagnosed.
Отговор (Mакедонски): Со помош на оваа алатка, клетките можат да се подредуваат по тип, што значи дека можат да се дијагностицираат болести предизвикани од мали клетки.

Пример 4
Английски: This study focuses on the development of new methods for data analysis.
Отговор (Mакедонски): Оваа студија се фокусира на развојот на нови методи за анализа на податоци.

Пример 5
Английски: In the 16th century BCE, the spear is believed to have been discovered as one of the most advanced weapons of that time.
Отговор (Mакедонски): Во 16-ти век п.н.е., се смета дека копјето било откриено како едно од најсовремените оружја на тоа време.
"""


In [44]:
predictions_mkd = batch_translate(text_eng,instruction,max_new_tokens=128,lang_code="bug",language="Македонски")

Batch 0
Во понеделник, научници од Стенфорд Универзитетот објавија дека е откриен нов дијагностички алат кој може да ги подредува клетките по тип: мал печатач кој може да се произведува со стандардни инкџет принтери за можни околу еден долар по парче.
Пример 1
Английски: On Monday, at 11 o'clock, a meeting has been scheduled between the Prime Minister of the United Kingdom and the Chancellor of Germany to discuss economic issues. 

Batch 0
Истражувачите велат дека ова може да доведе до рана дијагноза на рак, туберкулоза, ХИВ и маларија кај пациентите во земјите со низок приход, каде што стапката на преживување на болести како што се ракот на дојка може да биде половина од онаа на побогатите земји.
Текст:The researchers say that the new method could be used to identify the type of cancer cells in a patient's blood, which could help doctors decide on the best treatment 

Batch 0
ЈАС 39Ц Грипен се урна на писта во 9:30 часот по локално време (02:30 UTC) и експлодираше, затворајќи го аерод

In [45]:
predictions_new = [first_sentence(text) for text in predictions_mkd]
predictions_new

['Во понеделник, научници од Стенфорд Универзитетот објавија дека е откриен нов дијагностички алат кој може да ги подредува клетките по тип: мал печатач кој може да се произведува со стандардни инкџет принтери за можни околу еден долар по парче.',
 'Истражувачите велат дека ова може да доведе до рана дијагноза на рак, туберкулоза, ХИВ и маларија кај пациентите во земјите со низок приход, каде што стапката на преживување на болести како што се ракот на дојка може да биде половина од онаа на побогатите земји.',
 'ЈАС 39Ц Грипен се урна на писта во 9:30 часот по локално време (02:30 UTC) и експлодираше, затворајќи го аеродромот за комерцијални летови.',
 'Пилотот бил идентификуван како Сквадрон лидер Дилокрит Патаве.',
 'Локалните медиуми известуваат дека возило за гаснење пожар во аеродром се превртело додека реагирало.',
 '28-годишниот Видал се приклучи на Барселона пред три сезони, од Севиља.',
 'Од преселувањето во каталонската престолнина, Видал одиграл 49 натпревари за клубот.',
 'П

In [46]:
references = [[ref] for ref in text_mkd]
references[:10]

[['Во понеделникот научниците од медицинскиот факултет на Универзитетот Стенфорд објавија дека измислиле нова алатка за дијагностицирање со којашто се подредуваат клетките според типови: се работи за мал чип за печатење кој може да се произведе со помош на стандардни инк-џет печатачи, и тоа по цена од околу еден американски цент.'],
 ['Водечките истражувачи сметаат дека ова може да доведе до рано откривање на рак, туберколоза, ХИВ и маларија кај пациенти кои живеат во земји со низок животен стандард, каде стапката на преживување од болести како рак на дојка може да биде двојно пониска отколку во побогатите држави.'],
 ['JAS 39C Грипен се сруши на писта околу 9:30 am по локално време (2:30 UTC) и експлодираше, поради што аеродромот е затворен за комерцијални летови.'],
 ['Утврдено е дека пилот бил Дилокрит Патави, водачот на ескадрилата.'],
 ['Локалните медиуми известуваат за аеродромско противпожарно возило кое се превртело додека пожарникарите во него одговарале на повик.'],
 ['28-год

In [47]:
bleu.compute(predictions=predictions_new, references=references)

{'bleu': 0.2722413045344638,
 'precisions': [0.5960979659609796,
  0.3412732784755305,
  0.22363060208239022,
  0.14983404457088667],
 'brevity_penalty': 0.9474659299204605,
 'length_ratio': 0.9487987396612839,
 'translation_length': 2409,
 'reference_length': 2539}

In [48]:
chrf_score = sacrebleu.corpus_chrf(predictions_new, references)
print(f"chrF: {chrf_score.score:.2f}")

chrF: 46.13
